# 3분커리 임베딩 워커 (bge-m3)

서비스 서버(`worker_config.json`의 `base_url`)로 **먼저 접속**해서 관심 문장의 임베딩 요청을 가져가 처리하고 결과를 돌려준다. Kaggle은 외부에서 들어오는 연결을 받을 수 없기 때문이다.
요청이 `idle_exit_min`분 동안 없으면 스스로 종료해 GPU 쿼터를 아낀다. 워커가 없으면 서비스는 사전 검색으로 후퇴한다.

입력(gpuhub가 자동 연결): `worker.py`, `app_profiles.jsonl`, `worker_config.json`(주소·토큰·유휴/최대 시간)

In [ ]:
import torch
print('GPU 사용 가능:', torch.cuda.is_available())

In [ ]:
!pip install -q sentence-transformers

In [ ]:
import os, glob, shutil
need = ['worker.py', 'app_profiles.jsonl', 'worker_config.json', 'curated_doc_embeddings.npz']
search = ['search_chunks.jsonl', 'search_vectors.npy', 'intent_exemplars.json']   # 없으면 질의 전체 분석 없이 동작
if os.path.exists('/kaggle/input'):
    for f in need + search:
        hits = glob.glob(f'/kaggle/input/**/{f}', recursive=True)
        if not os.path.exists(f) and hits:
            shutil.copy(hits[0], f)
missing = [f for f in need if not os.path.exists(f)]
assert not missing, f'없는 파일: {missing}'
print('입력 준비 완료')

In [ ]:
import worker
worker.main('worker_config.json', 'app_profiles.jsonl')